# Clasificación del perfil financiero

---

Este modelo, permite hacer el analisis de datos y Machine Learning de la aplicación. Facilitando:
Evaluación del perfil de salud financiera: Permite aplicar las reglas del negocio sobre las variables financieras, como en el nivel de endeudamiento, los ingresos y los habitos de ahorro, para así clasificar al usuario en diferenetes perfiles Como:
 * Saludable (Ahorrador)
 * Equilibrado
 * Riesgo Alto (Sobreendeudado)


---

Importando las bibliotecas que vamos a utilizar:
* Pandas: Sirve para crear y manipular datos, además de leer archivos en CSV
* Numpy: Es una librería que realiza cálculos matemáticos y arreglos numericos
* Matplotlib: Permite a pyton crear graficos de distintos tipos
* seaborn: Hace graficos más atractivos

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Carga de datos y limpieza inicial.  Es importante cargas los datos, en este caso usaremos un DataFrame estructurado en Formato tipo JSON

In [8]:
# Muestra extraída directamente de banco_transacciones.csv
data_app = {
    'Id_Evento': ['61864c43', 'd9ec4024', '95efb8ba', '39b30404', '11266314'],
    'Id_Cliente': ['CLI-10025', 'CLI-10013', 'CLI-10000', 'CLI-10098', 'CLI-10061'],
    'Numero_Cuenta': ['MX-ACC-132097', 'MX-ACC-613153', 'MX-ACC-164820', 'MX-ACC-841040', 'MX-ACC-216421'],
    'Nombre_Cliente': ['Manuel', 'Javier', 'Luis', 'Alejandro', 'Sofia'],
    'Apellido_Cliente': ['Flores Sánchez', 'Rodríguez Gómez', 'Gómez Martínez', 'Martínez Ramírez', 'Flores Ramírez'],
    'Edad': [53, 34, 56, 33, 54],
    'Genero': ['M', 'M', 'M', 'M', 'F'],
    'Lugar_Registro': ['Monterrey', 'Querétaro', 'Monterrey', 'Guadalajara', 'Mérida'],
    'Ingreso_Mensual_Cliente': [41019.00, 15206.22, 20805.77, 50378.28, 18022.30],
    'Ahorro_Actual_Cliente': [23289.91, 11029.05, 0.00, 27819.55, 16724.07],
    'Fecha_Hora': ['2026-01-01 02:08:00', '2026-01-01 02:44:00', '2026-01-01 04:46:00', '2026-01-01 05:06:00', '2026-01-01 07:40:00'],
    'Tipo_Transaccion': ['Ingreso', 'Egreso', 'Egreso', 'Egreso', 'Egreso'],
    'Id_Categoria': ['CAT-120', 'CAT-104', 'CAT-127', 'CAT-126', 'CAT-106'],
    'Categoria_Transaccion': ['Nómina/Ingresos', 'Bus', 'Supermercado', 'Streaming', 'Comida rápida'],
    'Descripcion_Transaccion': ['Nómina', 'Bus', 'Costco', 'Amazon Prime', 'Burger King'],
    'Cantidad_Monto': [20509.50, 6.00, 758.00, 99.00, 450.00],
    'Moneda_Divisa': ['MXN', 'MXN', 'MXN', 'MXN', 'MXN'],
    'Metodo_Pago': ['Transferencia', 'Transferencia', 'En efectivo', 'Tarjeta de crédito', 'Tarjeta de crédito'],
    'Tiene_Tarjeta_Credito': ['Sí', 'Sí', 'Sí', 'Sí', 'Sí'],
    'Tarjeta_Credito_Estatus': ['Adeudo', 'Adeudo', 'Al corriente', 'Al corriente', 'Al corriente'],
    'Buro_Credito_Score': [799, 742, 487, 808, 839],
    'Estatus_Impuestos_Buro': ['Al corriente', 'Al corriente', 'Al corriente', 'Al corriente', 'Adeudo']
}



In [9]:
# Convertimos el diccionario a un DataFrame de Pandas
df = pd.DataFrame(data_app)

Transformacion de datos : Es importante limpiar, dar formato y adaptar los datos crudos para que sean procesables y comparables.
Ingeniería de Atributos: Permite crear nuevas variables a partir de los datos existentes para darle "conocimiento o contexto financiero" al modelo

In [10]:
# Convertimos la columna de texto a formato fecha real con Pandas
df['Fecha_Hora'] = pd.to_datetime(df['Fecha_Hora'])

# Extraemos si la transacción ocurrió en fin de semana (Sábado/Domingo)
df['dia_semana'] = df['Fecha_Hora'].dt.dayofweek
df['es_fin_de_semana'] = np.where(df['dia_semana'] >= 5, 1, 0)  # Uso de NumPy para la condición

# Calculamos el porcentaje que representa el gasto frente al ingreso
df['ratio_gasto_ingreso'] = (df['Cantidad_Monto'] / df['Ingreso_Mensual_Cliente']).round(4)

Preprocesamiento (Convertir textos a números y Escalar)

In [11]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# A) LabelEncoder: Convierte descripciones en texto a números enteros
# Ejemplo: 'Costco' -> 0, 'Amazon Prime' -> 1
le_desc = LabelEncoder()
df['desc_encoded'] = le_desc.fit_transform(df['Descripcion_Transaccion'])

# B) StandardScaler: Ajusta montos grandes para que no distorsionen al modelo
scaler = StandardScaler()
df['monto_scaled'] = scaler.fit_transform(df[['Cantidad_Monto']])
df['score_scaled'] = scaler.fit_transform(df[['Buro_Credito_Score']])

Separación de Datos (Train / Test)

In [12]:
from sklearn.model_selection import train_test_split

# Filtramos solo las compras/gastos (Egresos)
df_gastos = df[df['Tipo_Transaccion'] == 'Egreso'].copy()

# 'X' son las variables que el modelo analiza para aprender
X = df_gastos[['monto_scaled', 'desc_encoded', 'es_fin_de_semana', 'ratio_gasto_ingreso', 'score_scaled']]

# 'y' es la respuesta correcta (la categoría real del gasto)
y = df_gastos['Categoria_Transaccion']

# Dividimos en 80% para entrenar y 20% para evaluar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Crear y Entrenar el Modelo de Inteligencia Artificial: Se usará Bosque aleatorio

In [13]:
from sklearn.ensemble import RandomForestClassifier

# Creamos el modelo (Bosque Aleatorio)
modelo = RandomForestClassifier(n_estimators=100, random_state=42)

# Lo entrenamos pasando las entradas (X) y las respuestas esperadas (y)
modelo.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Evaluación del Modelo

In [14]:
from sklearn.metrics import classification_report

# El modelo hace predicciones sobre datos de prueba que nunca había visto
y_pred = modelo.predict(X_test)

# Generamos un reporte impreso para ver qué tan bien clasificó los gastos
print("--- Reporte de Desempeño del Modelo ---")
print(classification_report(y_test, y_pred, zero_division=0))

--- Reporte de Desempeño del Modelo ---
               precision    recall  f1-score   support

Comida rápida       0.00      0.00      0.00       0.0
 Supermercado       0.00      0.00      0.00       1.0

     accuracy                           0.00       1.0
    macro avg       0.00      0.00      0.00       1.0
 weighted avg       0.00      0.00      0.00       1.0

